## **CRAWLING DATA**

## Crawling Data Detik.com

Pada bagian ini, kita akan mempraktikkan tahapan pengumpulan data (*web crawling*) dengan mengambil data judul dan isi berita olahraga terbaru yang di bagi menjadi 2 kelas yaitu **SPORT** dan **FINANCE** dari portal **Detik.com**. 

Untuk mengekstrak data ini, kita menggunakan tiga *library* utama:
* **Requests:** Untuk mengirim permintaan (HTTP Request) ke server web dan mengunduh kode HTML mentahnya.
* **BeautifulSoup:** Untuk memecah dan menstrukturkan HTML tersebut agar elemen spesifik (seperti teks judul dan tautan) mudah disaring.
* **Pandas:** Untuk menyusun data hasil ekstraksi menjadi format tabel (*DataFrame*) yang rapi.
* **Trafilatura** Agar secara otomatis masuk ke setiap tautan dan mengekstrak teks isi beritanya

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import trafilatura

kategori_list = ['sport', 'finance']
target_per_kategori = 100
data_berita = []

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}

for kategori in kategori_list:
    print(f"\n=== Memulai Crawling Kategori: {kategori.upper()} ===")
    halaman = 1
    terkumpul = 0
    
    while terkumpul < target_per_kategori:
        url = f'https://{kategori}.detik.com/indeks?page={halaman}'
            
        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            articles = soup.find_all('article')
            if not articles:
                articles = soup.select('.list-content article, .media__text')
            
            if not articles:
                print(f"Halaman {halaman} benar-benar kosong. Berhenti di halaman ini.")
                break
                
            for article in articles:
                if terkumpul >= target_per_kategori:
                    break
                    
                title_tag = article.find('h3') or article.find('h2')
                link_tag = article.find('a')
                
                if title_tag and link_tag and 'href' in link_tag.attrs:
                    judul = title_tag.get_text(strip=True)
                    tautan = link_tag['href']
                    
                    downloaded = trafilatura.fetch_url(tautan)
                    if downloaded:
                        isi_berita = trafilatura.extract(downloaded)
                    else:
                        isi_berita = "Gagal mengekstrak isi"
                    
                    # Menyimpan data dengan membuang kolom tautan dan menjadikannya Label Kategori
                    data_berita.append({
                        'Label Kategori': kategori.upper(),
                        'Judul Berita': judul,
                        'Isi Berita': isi_berita
                    })
                    terkumpul += 1
                    
            print(f"[{kategori.upper()}] Halaman {halaman} selesai. Terkumpul: {terkumpul}/{target_per_kategori}")
            
        except Exception as e:
            print(f"Gagal mengakses {url}: {e}")
            
        time.sleep(1.5) 
        halaman += 1

df_berita = pd.DataFrame(data_berita)
nama_file = 'data_berita_sport_finance.csv'
df_berita.to_csv(nama_file, index=False)

print(f"\nProses selesai! Total data: {df_berita.shape[0]} baris.")
display(df_berita.head())


=== Memulai Crawling Kategori: SPORT ===
[SPORT] Halaman 1 selesai. Terkumpul: 20/100
[SPORT] Halaman 2 selesai. Terkumpul: 40/100
[SPORT] Halaman 3 selesai. Terkumpul: 60/100
[SPORT] Halaman 4 selesai. Terkumpul: 80/100
[SPORT] Halaman 5 selesai. Terkumpul: 100/100

=== Memulai Crawling Kategori: FINANCE ===
[FINANCE] Halaman 1 selesai. Terkumpul: 20/100
[FINANCE] Halaman 2 selesai. Terkumpul: 40/100
[FINANCE] Halaman 3 selesai. Terkumpul: 60/100
[FINANCE] Halaman 4 selesai. Terkumpul: 80/100
[FINANCE] Halaman 5 selesai. Terkumpul: 100/100

Proses selesai! Total data: 200 baris.


,Label Kategori,Judul Berita,Isi Berita
0,SPORT,"Sempat Ada Banjir, Asian Games 2026 di Jepang ...",Kota Nagoya di Jepang selaku tuan rumah Asian ...
1,SPORT,"Debut di Asian Games, Ubed Tak Sabar Rebut Ema...",Moh. Zaki Ubaidillah akan menjalani debutnya d...
2,SPORT,Men's World Tennis Championship: Rifqi dan Raf...,"Dua petenis Indonesia, Muhammad Rifqi Fitriadi..."
3,SPORT,Kevin Sanjaya Antusias Lihat Semangat Talenta ...,Audisi Umum PB Djarum 2026 sedang berlangsung ...
4,SPORT,Atlet Muda Modern Pentathlon Punya Kesempatan ...,Modern Pentathlon akan dipertandingkan dalam m...


Data tabular di atas merupakan hasil akhir dari proses pengumpulan informasi mentah pada halaman indeks. Setelah dikumpulkan dalam format tabel seperti ini, dataset sudah siap untuk dibersihkan dan diproses lebih lanjut pada tahap prapemrosesan teks (*text preprocessing*) dalam siklus Web Mining.